[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohsennasab/python-fundamentals-hh/blob/main/notebooks/08_landuse-data/08_01_land_cover_impervious.ipynb)


# Module 8: Land Cover and Impervious Surface Data for H&H Modeling
## Watershed Land Cover Summaries and Percent Impervious with NLCD

### Welcome!
This module teaches you how to work with official USGS/MRLC land cover and impervious surface data for watershed hydrology. You will learn the difference between a categorical land cover raster and a continuous percent-impervious raster, calculate watershed land cover summaries and percent impervious correctly, and compare land cover between two years to see how a watershed has changed. The notebook fetches its own raster data from a free government web service, so there is nothing to download or upload beyond the watershed boundary file. Along the way, this module walks through a real data mistake that is easy to make and easy to miss, so you can recognize the same pattern in your own work.

### What You'll Work Through Today:
- Understand the difference between categorical and continuous rasters, and why it matters for calculations
- Fetch NLCD rasters automatically from the MRLC web service, clipped to your watershed's own extent
- Load a watershed boundary and reproject it to match a raster's coordinate system
- Clip an NLCD land cover raster to a watershed and summarize classes by area and percent
- Clip an NLCD percent impervious raster and calculate a correct, defensible watershed-average value
- Use the impervious descriptor product to separate roads from other developed surfaces
- Compare land cover and imperviousness between two years to quantify change
- Export model-ready summary tables and clipped rasters

### Module Structure:
1. **Mental Models** - Land cover, land use, and the categorical/continuous distinction
2. **The Data Source** - NLCD, MRLC access options, and today's scenario
3. **Workspace Setup** - Libraries and the watershed boundary
4. **Watershed, Data Fetch, and CRS** - Loading the study area, fetching its rasters, and getting the CRS direction right
5. **Land Cover Workflow** - Clip, count, and summarize by class
6. **Percent Impervious Workflow** - The correct calculation, and a real mistake worth knowing about
7. **Impervious Descriptor** - Roads versus other built surfaces
8. **Change Analysis** - 2001 versus 2021
9. **Export** - Model-ready tables and clipped rasters
10. **Beyond NLCD** - C-CAP and local data refinements

### Prerequisites
This module assumes you have completed Module 1 (Python fundamentals), Module 3 (vector data and CRS), and Module 4 (raster fundamentals: pixels, resolution, clipping). Module 7 (SSURGO soils) is helpful background, both because its output pairs directly with this module's land cover table for curve number work and because it introduced the web request pattern this module reuses, but it is not required.


## Using AI in This Module

This module works with two very different kinds of rasters in the same lesson: a categorical land cover raster and a continuous percent-impervious raster. If a step feels confusing, ask your AI assistant:

- *"Why can't I average land cover class codes the way I would average a percentage?"*
- *"I clipped a raster to my watershed and the pixel count looks wrong. What should I check?"*
- *"My percent-impervious result changed a lot after I fixed my NoData value. Why would that happen?"*

As always, review what the AI suggests before you run it. This module in particular is built around a real mistake that is easy to make silently. Use your assistant to help you reason through why the mistake happens, not just to get an answer.


## Part 1: Mental Models - Land Cover, Land Use, and Two Kinds of Rasters 🧠

### Land Cover Is Not the Same as Land Use

**Land cover** is what physically covers the ground: forest, pavement, water, grass. **Land use** is how people use that land: a park and a residential yard can have the exact same grass land cover but very different land use. H&H modeling almost always needs land cover, because runoff and infiltration respond to what is physically on the ground, not to zoning. Every product in this module is a land cover product.

### A Quick Raster Recap

Module 4 introduced rasters as grids of cells, each holding a value, with a resolution, a coordinate reference system, and a NoData convention. This module builds directly on that. If any of those words feel unfamiliar, a quick look back at Module 4 will help before continuing here.

### Categorical vs. Continuous Rasters: The Idea This Module Is Built On

This is the single most important distinction in this module, so it is worth sitting with for a moment.

- A **categorical raster** stores a class code in every cell. NLCD land cover is categorical: a cell with the value 42 means "evergreen forest." The number 42 is a label, not a quantity. Averaging class codes together (adding 42 and 21 and dividing by two) produces a meaningless number. You can only count how many cells fall into each class.
- A **continuous raster** stores a measured quantity in every cell. The NLCD fractional impervious surface product is continuous: a cell with the value 37 means an estimated 37 percent of that 30 meter cell is impervious. Averaging these values together is exactly the right thing to do; that is what the product is for.

| | Categorical | Continuous |
|---|---|---|
| Example in this module | Land Cover, Impervious Descriptor | Fractional Impervious Surface |
| What a cell value means | A class code | A measured percentage |
| Correct summary | Count cells per class, convert to area | Average the values, weighted by area |
| Wrong summary | Averaging the codes | Treating it like a category |

### The Three NLCD Products Used in This Module

| Product | Type | What It Tells You |
|---|---|---|
| Land Cover | Categorical | Which land cover class each cell belongs to (forest, developed, water, and so on) |
| Fractional Impervious Surface | Continuous | The estimated percent impervious surface in each cell |
| Impervious Descriptor | Categorical | Whether an impervious cell is a road, or another kind of built surface |

The rule of thumb this module follows: use Land Cover for class summaries, use Fractional Impervious Surface whenever you need a percent impervious number, and use the Impervious Descriptor only when you specifically need to separate roads from buildings. Do not estimate percent impervious just by counting developed land cover classes; a "Developed, Open Space" cell is nowhere near 100 percent impervious, and the fractional product exists precisely so you do not have to guess.

### Why This Matters in H&H Work

Land cover and imperviousness feed directly into:

- **Curve numbers.** Module 4 built a composite curve number assuming a single hydrologic soil group. A real curve number workflow combines land cover (this module) with hydrologic soil group (Module 7) for each pixel.
- **Manning's roughness.** Land cover class is a common basis for assigning roughness zones in a hydraulic model.
- **HEC-HMS and HEC-RAS parameters.** Percent impervious is a direct input to many urban hydrology loss methods.
- **Existing versus future comparisons.** Land cover from two different years is exactly how a "what changed" analysis is built, which this module demonstrates directly.

### Key Terms

| Term | Meaning |
|---|---|
| Class code | A whole number representing one land cover category (categorical rasters only) |
| Legend | The lookup table connecting each class code to a name and a display color |
| Percent impervious | The estimated fraction of a cell's area covered by impervious surface, 0 to 100 |
| NoData | A raster's way of marking cells with no valid value; the correct NoData value differs by product, as this module will show |
| Equal-area projection | A coordinate system where every cell represents the same true ground area, which is what makes area and percentage math meaningful |
| Cell area | For a 30 meter NLCD cell: 30 x 30 = 900 square meters, which is 0.2224 acres |


## Part 2: The Data Source - NLCD and How to Get It 🌐

### About NLCD

The National Land Cover Database (NLCD) is the official USGS/MRLC land cover product for the United States, produced roughly every two to three years since 2001 at 30 meter resolution. This module uses the **2001** and **2021** releases, which together give a 20 year window for change analysis.

You may also come across the newer **Annual NLCD** product line, which classifies land cover for every calendar year from 1985 to the present. It is distributed through a cloud archive on Amazon Web Services (USGS's `usgs-landcover` S3 bucket), organized as tiles and mosaics by year. That archive is configured as a **requester-pays bucket**, which means every download is billed to the requester's own AWS account, and AWS refuses to serve a requester-pays bucket to an anonymous, unauthenticated request. In practice that means every student would need a personal AWS account with billing enabled before they could pull a single file, which does not fit a free, no-installation course. This module uses the standard NLCD releases instead, through a service that needs no account and no payment at all, described next.

### How This Notebook Gets Its Data

This notebook does not ask you to download or upload any raster files. Instead, it calls the official **MRLC Web Coverage Service (WCS)**, a free, no-login web service, and asks it for a small GeoTIFF clipped to your study watershed's own extent, plus a small buffer. This is the same kind of request you saw for SSURGO in Module 7, just for a different agency's service. The result is a handful of small files, typically a few hundred kilobytes each, saved locally in your Colab session.

If your watershed is very large, note that a WCS request that covers too much area can be slow or may be rejected by the service; the buffer used in this notebook is intentionally modest for that reason.

### Access Options

MRLC and USGS offer several official ways to get NLCD data. This module implements one of them (the scripted WCS request) and lists the others so you know where to look for different needs.

| Access Method | Best For |
|---|---|
| MRLC Data Portal | Manually downloading a full GeoTIFF for a product and year |
| MRLC Viewer / Mosaic Download | Selecting an area of interest and downloading just that area, no scripting required |
| MRLC Web Coverage Service (WCS) | Scripted, reproducible downloads clipped to a bounding box; used in this module |
| MRLC Web Map Service (WMS) | Visualization in GIS software, not for retrieving raw values |
| EarthExplorer | Official USGS archive access, tile-based |
| ScienceBase | Citable, versioned data releases |
| USGS cloud storage (S3), Annual NLCD | Cloud-native access to the newer yearly product line; requester-pays, needs an AWS account with billing |

### What Happens If the Live Service Is Unavailable

Web services occasionally have brief outages. If the live WCS request fails, this notebook automatically falls back to a set of pre-clipped rasters hosted in the course's own GitHub repository, covering the same 19-watershed study area used throughout this course. You do not need to do anything differently; the fallback happens automatically and prints a message when it does.

### Today's Engineering Scenario

You are assessing development impacts in the Crow Creek area south of Cheyenne, Wyoming. A county stormwater master plan update needs to know how developed each HUC-12 watershed is today, how much of it is effectively impervious, and how that has changed over the last 20 years. Field survey is not budgeted for this scale of screening. NLCD is the appropriate official source for a first-pass, county-wide answer.


## Part 3: Workspace Setup 🛠️

Google Colab already includes `rasterio`, `geopandas`, `numpy`, `pandas`, `requests`, and `matplotlib`, so nothing needs to be installed.


In [ ]:
# Tabular data
import pandas as pd
import numpy as np

# Spatial data
import geopandas as gpd
import rasterio
import rasterio.mask

# Web requests, for fetching NLCD rasters directly from MRLC
import requests

# Plotting
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

print("Libraries imported. Ready to work with land cover data.")


### Uploading the Watershed Boundary

Upload the same watershed boundary file used in Modules 3, 5, and 7. The five NLCD rasters are not uploaded; the notebook fetches those for you in Part 4.


In [ ]:
from google.colab import files

print("Please upload this file:")
print("1. NHD__Watershed_Boundaries_HUC_12_Selected.zip")
print()
print("Click 'Choose Files' below and select it.")

uploaded = files.upload()

print(f"\nUploaded {len(uploaded)} file(s):")
for filename in uploaded.keys():
    print(f"   - {filename}")


## Part 4: Watershed, Data Fetch, and CRS 🌍

### Load the Watershed

This module reuses the same Wyoming HUC-12 watershed from Modules 3, 5, and 7: watershed `101900090108`, "Town of South Greeley."


In [ ]:
watersheds = gpd.read_file(
    'zip://NHD__Watershed_Boundaries_HUC_12_Selected.zip'
)

target_huc = '101900090108'
target_watershed = watersheds[watersheds['HUC12'] == target_huc]

print(f"Target watershed: {target_watershed['Name'].iloc[0]}")
print(f"Reported area: {target_watershed['AreaAcres'].iloc[0]:,.0f} acres")
print(f"Watershed CRS: {target_watershed.crs}")


### Fetching NLCD Rasters for This Watershed

Now that a watershed is selected, build a bounding box around it and use that box to request exactly the area needed from MRLC. NLCD is served in a Conterminous US Albers Equal-Area projection (EPSG:5070), so the watershed is reprojected into that CRS before its bounds are used, even before any raster has been opened.


In [ ]:
# NLCD's native CRS, used by the MRLC web service
NLCD_CRS = "EPSG:5070"

# Reproject just for the bounding box calculation; the full reprojection
# used for clipping happens later, once we can check it against the
# raster we actually receive
target_bbox_crs = target_watershed.to_crs(NLCD_CRS)
minx, miny, maxx, maxy = target_bbox_crs.total_bounds

# A small buffer keeps the watershed away from the very edge of the
# fetched raster
buffer_m = 500
minx, miny, maxx, maxy = minx - buffer_m, miny - buffer_m, maxx + buffer_m, maxy + buffer_m

print(f"Watershed bounding box in {NLCD_CRS}, with a {buffer_m} m buffer:")
print(f"  X: {minx:.0f} to {maxx:.0f}")
print(f"  Y: {miny:.0f} to {maxy:.0f}")


### A Reusable Fetch Function

This function sends one WCS request for one product, saves the result, and writes the correct NoData value into the file (0 for land cover, 127 for the impervious and descriptor products, as explained in Part 6). If the live request fails for any reason, it automatically retries against a pre-clipped copy of the same product hosted in this course's GitHub repository, so a temporary MRLC outage does not stop the lesson.


In [ ]:
WCS_URL = "https://www.mrlc.gov/geoserver/wcs"

# A pre-clipped fallback copy of every product used in this module,
# covering the same 19-watershed study area, hosted alongside this notebook
FALLBACK_BASE_URL = (
    "https://raw.githubusercontent.com/mohsennasab/python-fundamentals-hh/"
    "main/notebooks/08_landuse-data/data"
)


def fetch_nlcd_raster(filename, coverage_id, nodata_value, minx, miny, maxx, maxy):
    """Fetch one NLCD product clipped to a bounding box, with a fallback.

    Tries the live MRLC Web Coverage Service first. If that request fails,
    falls back to a pre-clipped copy of the same product from this
    course's GitHub repository.
    """
    params = {
        "service": "WCS",
        "version": "2.0.1",
        "request": "GetCoverage",
        "coverageId": coverage_id,
        "subset": [f"X({minx},{maxx})", f"Y({miny},{maxy})"],
        "format": "image/geotiff",
    }

    try:
        response = requests.get(WCS_URL, params=params, timeout=60)
        if response.status_code != 200 or len(response.content) < 1000:
            raise ValueError(f"WCS returned status {response.status_code}")
        with open(filename, 'wb') as f:
            f.write(response.content)
        print(f"  fetched from MRLC WCS ({len(response.content):,} bytes)")
    except Exception as e:
        print(f"  live WCS request failed ({e}); trying the course repository fallback...")
        fallback_url = f"{FALLBACK_BASE_URL}/{filename}"
        response = requests.get(fallback_url, timeout=60)
        if response.status_code != 200 or len(response.content) < 1000:
            raise RuntimeError(
                f"Both the live WCS request and the fallback failed for {filename}. "
                f"Fallback status code: {response.status_code}. Check your internet "
                f"connection and try again."
            )
        with open(filename, 'wb') as f:
            f.write(response.content)
        print(f"  fetched from course repository fallback ({len(response.content):,} bytes)")

    # Write the correct NoData value for this specific product. The raw
    # download's own metadata is not trustworthy here (see Part 6), so
    # this is set deliberately rather than left to whatever the source
    # file happened to declare.
    with rasterio.open(filename) as src:
        data = src.read(1)
        profile = src.profile.copy()
        # Land cover files carry an embedded color table; keep it so the
        # file still displays with official NLCD colors in GIS software
        try:
            colormap = src.colormap(1)
        except ValueError:
            colormap = None

    profile.update(nodata=nodata_value, compress='lzw')
    with rasterio.open(filename, 'w', **profile) as dst:
        dst.write(data, 1)
        if colormap is not None:
            dst.write_colormap(1, colormap)


print("fetch_nlcd_raster() is ready.")


### Fetching All Five Products

This is the step that replaces a manual download or upload entirely. Each product is a separate request to the same service.


In [ ]:
# filename -> (WCS coverage ID, correct NoData value for that product)
nlcd_products = {
    'nlcd_2001_land_cover.tif': ('mrlc_download__NLCD_2001_Land_Cover_L48', 0),
    'nlcd_2021_land_cover.tif': ('mrlc_download__NLCD_2021_Land_Cover_L48', 0),
    'nlcd_2001_impervious.tif': ('mrlc_download__NLCD_2001_Impervious_L48', 127),
    'nlcd_2021_impervious.tif': ('mrlc_download__NLCD_2021_Impervious_L48', 127),
    'nlcd_2021_impervious_descriptor.tif': ('mrlc_download__NLCD_2021_Impervious_descriptor_L48', 127),
}

print("Fetching NLCD rasters for this watershed's extent...\n")

for filename, (coverage_id, nodata_value) in nlcd_products.items():
    print(f"{filename}:")
    fetch_nlcd_raster(filename, coverage_id, nodata_value, minx, miny, maxx, maxy)

print("\nAll five NLCD rasters are ready.")


### Check the Raster's CRS

Open the 2021 land cover raster and look at its coordinate reference system.


In [ ]:
lc_2021_path = 'nlcd_2021_land_cover.tif'

with rasterio.open(lc_2021_path) as src:
    lc_crs = src.crs
    lc_res = src.res
    lc_nodata = src.nodata

print(f"Land cover raster CRS: {lc_crs}")
print(f"Resolution: {lc_res[0]} x {lc_res[1]} meters")
print(f"Declared NoData value: {lc_nodata}")


### Why NLCD Uses an Equal-Area Projection

The land cover raster is in a Conterminous US Albers Equal-Area projection, not the geographic latitude/longitude system. This is deliberate. In an equal-area projection, every cell covers the same true ground area (900 square meters for a 30 meter NLCD cell), everywhere in the country. That is exactly what area and percentage calculations need. A geographic CRS would distort cell area depending on latitude, which would quietly bias every area calculation in this module.

### Reproject the Vector, Not the Raster

The watershed boundary is in a different CRS than the land cover raster. There are two ways to fix this, and only one of them is a good idea:

- **Reproject the vector watershed boundary to match the raster's CRS.** This is exact. A polygon boundary is a mathematical shape, and reprojecting it just recalculates its coordinates.
- **Reproject the raster to match the vector's CRS.** This resamples every pixel, which for a categorical raster like land cover means class codes get blended or reassigned at cell edges. For a percent-impervious raster it introduces a similar distortion. Either way, you would be changing the data to avoid changing the boundary, which is backwards.

This module always reprojects vectors to match the raster CRS, never the other way around.

🤖 **Try asking your AI assistant:** *"Why is it better to reproject a watershed boundary to match a raster's CRS, instead of reprojecting the raster to match the vector? What actually happens to the data in each case?"*


In [ ]:
# Reproject the watershed boundary to match the land cover raster's CRS
target_reprojected = target_watershed.to_crs(lc_crs)

print(f"Watershed CRS is now: {target_reprojected.crs}")

# Recompute area from the reprojected polygon, in acres and square miles
# EPSG:5070 uses meters, so area comes back in square meters
area_m2 = target_reprojected.geometry.iloc[0].area
area_acres = area_m2 / 4046.86
area_sqmi = area_acres / 640

print(f"Watershed area: {area_acres:,.0f} acres ({area_sqmi:.1f} square miles)")


## Part 5: Land Cover Workflow - From Pixels to a Class Table 📊

### Step 1: Clip the Raster to the Watershed

`rasterio.mask.mask` keeps only the pixels inside the watershed polygon and fills everything else with the raster's NoData value.


In [ ]:
watershed_geom = [target_reprojected.geometry.iloc[0].__geo_interface__]

with rasterio.open(lc_2021_path) as src:
    lc_clipped, lc_transform = rasterio.mask.mask(
        src, watershed_geom, crop=True, nodata=0
    )

lc_array = lc_clipped[0]
print(f"Clipped array shape: {lc_array.shape}")
print(f"Total cells in clipped array: {lc_array.size}")


### Step 2: Count Pixels by Class

This raster is categorical, so counting is the correct operation. `numpy.unique` with `return_counts=True` gives every class code present and how many cells hold it. Class code 0 is NLCD's NoData value; no real land cover class uses 0, so it is safe to exclude it here.


In [ ]:
class_values, pixel_counts = np.unique(
    lc_array[lc_array != 0], return_counts=True
)

print(f"Found {len(class_values)} land cover classes in the watershed")
for code_val, count in zip(class_values, pixel_counts):
    print(f"  code {code_val}: {count} pixels")


### Step 3: The NLCD Legend

Convert class codes into names and colors using the official NLCD legend.


In [ ]:
nlcd_classes = {
    11: 'Open Water',
    21: 'Developed, Open Space',
    22: 'Developed, Low Intensity',
    23: 'Developed, Medium Intensity',
    24: 'Developed, High Intensity',
    31: 'Barren Land',
    41: 'Deciduous Forest',
    42: 'Evergreen Forest',
    43: 'Mixed Forest',
    52: 'Shrub/Scrub',
    71: 'Grassland/Herbaceous',
    81: 'Pasture/Hay',
    82: 'Cultivated Crops',
    90: 'Woody Wetlands',
    95: 'Emergent Herbaceous Wetlands',
}

# Official NLCD display colors, one per class code
nlcd_colors = {
    11: '#466b9f', 21: '#dec5c5', 22: '#d99282', 23: '#eb0000', 24: '#ab0000',
    31: '#b3ac9f', 41: '#68ab5f', 42: '#1c5f2c', 43: '#b5c58f', 52: '#ccb879',
    71: '#dfdfc2', 81: '#dcd939', 82: '#ab6c28', 90: '#b8d9eb', 95: '#6c9fb8',
}

print(f"Legend covers {len(nlcd_classes)} classes")


### Step 4: Convert Pixel Counts to Area and Percent

Each cell covers 900 square meters (30 m x 30 m). Convert to acres, then to a percentage of the watershed's total classified area.


In [ ]:
cell_area_m2 = 900
cell_area_acres = cell_area_m2 / 4046.86

total_pixels = pixel_counts.sum()

lc_summary = pd.DataFrame({
    'nlcd_code': class_values,
    'nlcd_class': [nlcd_classes.get(int(c), f'Unknown ({c})') for c in class_values],
    'pixel_count': pixel_counts,
})

lc_summary['area_acres'] = lc_summary['pixel_count'] * cell_area_acres
lc_summary['percent_watershed'] = 100 * lc_summary['pixel_count'] / total_pixels

lc_summary = lc_summary.sort_values('percent_watershed', ascending=False).reset_index(drop=True)

print(lc_summary[['nlcd_class', 'area_acres', 'percent_watershed']].round(1))


### Step 5: Sanity Check Against the Polygon Area

Compare the total classified area from the raster to the watershed's true polygon area from Part 4. They should be close; small differences come from how pixel edges approximate an irregular boundary.


In [ ]:
raster_total_acres = lc_summary['area_acres'].sum()

print(f"Watershed area from polygon geometry: {area_acres:,.0f} acres")
print(f"Watershed area from classified raster pixels: {raster_total_acres:,.0f} acres")
print(f"Difference: {abs(area_acres - raster_total_acres):,.0f} acres "
      f"({100*abs(area_acres - raster_total_acres)/area_acres:.1f}% of watershed area)")


A small difference here is normal and expected; it comes from approximating a curved, irregular watershed boundary with a grid of square 30 meter cells. A large difference (more than a few percent) would be worth investigating, since it could mean a CRS mismatch or a clipping error.

### Step 6: Map the Clipped Land Cover


In [ ]:
# Build a colormap that matches the official NLCD colors, in the same
# order as the class codes present in this watershed
present_codes = sorted(class_values.tolist())
colors_in_order = [nlcd_colors[c] for c in present_codes]
cmap = ListedColormap(colors_in_order)

# Map each raw class code to its position in present_codes, so matplotlib
# assigns the right color to the right code
code_to_index = {c: i for i, c in enumerate(present_codes)}
lc_display = np.vectorize(lambda v: code_to_index.get(v, -1))(lc_array)
lc_display = np.ma.masked_where(lc_array == 0, lc_display)

fig, ax = plt.subplots(figsize=(9, 8))
ax.imshow(lc_display, cmap=cmap, vmin=0, vmax=len(present_codes) - 1)

legend_patches = [
    Patch(facecolor=nlcd_colors[c], label=nlcd_classes[c]) for c in present_codes
]
ax.legend(handles=legend_patches, loc='center left', bbox_to_anchor=(1.0, 0.5),
          fontsize=9, frameon=True)

ax.set_title(f"2021 Land Cover\n{target_watershed['Name'].iloc[0]}", fontweight='bold')
ax.set_xticks([])
ax.set_yticks([])

plt.tight_layout()
plt.show()


### What This Output Tells Us

Grassland dominates this watershed, but the developed classes (open space through high intensity) together account for a meaningful share of the area. This is exactly the mixed rural/exurban character that makes this watershed a useful case for a stormwater screening study: it is not fully rural, and it is not fully urban.


## Part 6: Percent Impervious Workflow - A Real Mistake Worth Knowing About ⚠️

### First, the Wrong Way (Which Looks Completely Reasonable)

The fractional impervious surface raster is continuous, so the correct summary is an area-weighted average, not a class count. Here is a first attempt, following the exact same NoData-masking habit that worked perfectly well for the categorical land cover raster in Part 5.


In [ ]:
imp_2021_path = 'nlcd_2021_impervious.tif'

with rasterio.open(imp_2021_path) as src:
    imp_clipped_v1, _ = rasterio.mask.mask(
        src, watershed_geom, crop=True, nodata=0
    )

imp_array_v1 = imp_clipped_v1[0]

# Mask out NoData the same way we did for land cover: exclude 0
valid_v1 = imp_array_v1[imp_array_v1 != 0]
print(f"Valid pixel count (excluding 0): {valid_v1.size}")
print(f"Mean percent impervious: {valid_v1.mean():.2f}%")


### Something Is Wrong

Compare this valid pixel count to the land cover valid pixel count from Part 5. The land cover raster found several tens of thousands of valid pixels in the same watershed. This impervious result found far fewer. The two rasters are clipped to the exact same watershed at the exact same resolution, so their pixel counts should be nearly identical. They are not, and that is a signal to stop and investigate rather than report the number.

### Investigating: Is 0 Really "No Data" Here?

The land cover raster uses 0 as NoData because no real NLCD land cover class is ever coded 0; every legitimate value starts at 11. Is the same true for percent impervious?

Think about what a value of 0 in this raster means: 0 percent impervious. That is not missing data. It is a completely ordinary reading for a rural grassland pixel, and this watershed is 71 percent grassland. By excluding every pixel with a value of 0, the code above just excluded most of the genuinely rural, low-impervious part of the watershed, biasing the remaining average sharply upward toward the more developed pixels that happened to have a nonzero value.

### Finding the Real Answer

Official NLCD documentation defines a separate background value for this product, used only for cells outside the mapped extent (for example, cells over the ocean or outside the country): the value **127**. That is the true NoData sentinel for this product, not 0.


In [ ]:
with rasterio.open(imp_2021_path) as src:
    imp_clipped_v2, imp_transform = rasterio.mask.mask(
        src, watershed_geom, crop=True, nodata=127
    )

imp_array_v2 = imp_clipped_v2[0]

# Now check: does 127 actually appear anywhere in our study area?
n_background = (imp_array_v2 == 127).sum()
print(f"Pixels equal to 127 (true background) inside the watershed: {n_background}")

# And confirm the full value range present
valid_mask = imp_array_v2 != 127
print(f"Valid pixel count (excluding true background, 127): {valid_mask.sum()}")
print(f"Value range among valid pixels: {imp_array_v2[valid_mask].min()} to {imp_array_v2[valid_mask].max()}")


This watershed is entirely within the mapped extent, so there are zero true background pixels. Every single cell in the clipped raster is valid data, including the many cells with a real value of 0. The corrected valid pixel count should now match the land cover raster's pixel count from Part 5.

### The Corrected Calculation


In [ ]:
imp_valid = imp_array_v2[imp_array_v2 != 127].astype(float)

mean_impervious_wrong = valid_v1.mean()
mean_impervious_correct = imp_valid.mean()

print(f"Wrong approach (masked on 0):        {mean_impervious_wrong:.2f}% mean impervious")
print(f"Corrected approach (masked on 127):  {mean_impervious_correct:.2f}% mean impervious")
print(f"\nThe wrong approach overstated the watershed's imperviousness by "
      f"{mean_impervious_wrong - mean_impervious_correct:.1f} percentage points.")


### Why the Total Area Estimate Looked Fine, but the Percentage Did Not

If you had instead reported the total equivalent impervious acreage (percent times area, summed across all pixels), you might not have noticed a problem: the excluded pixels were mostly near zero, so they barely contributed to that sum either way. It is specifically the **average percentage** that gets distorted, because the denominator (how many pixels you are averaging over) shrank dramatically while the numerator barely changed. This is exactly why it is worth checking more than one derived number, and why a pixel-count mismatch between two supposedly identical clips is worth chasing down instead of ignoring.

### The Full Area-Weighted Calculation

Now that NoData is handled correctly, compute the full area-weighted percent impervious and the equivalent impervious area, following the same explicit, step-by-step style used for Ksat in Module 7.

One thing to notice: because every NLCD cell covers the same 900 square meters, the area-weighted average below comes out exactly equal to the simple mean printed above. So why bother with the longer form? Two reasons. First, it produces the equivalent impervious area as a byproduct, which is a useful engineering number on its own. Second, the explicit form is the one that stays correct when cells do not all represent the same area, for example when combining rasters of different resolutions or weighting partial cells along a boundary. Writing it out now builds the right habit for those situations.


In [ ]:
# Impervious area contributed by each valid cell
impervious_area_per_cell_m2 = (imp_valid / 100) * cell_area_m2

# Total impervious area across the watershed
total_impervious_area_m2 = impervious_area_per_cell_m2.sum()
total_impervious_area_acres = total_impervious_area_m2 / 4046.86

# Total valid (mapped) area across the watershed
total_valid_area_m2 = imp_valid.size * cell_area_m2
total_valid_area_acres = total_valid_area_m2 / 4046.86

# Watershed-average percent impervious, area-weighted
percent_impervious_2021 = 100 * total_impervious_area_m2 / total_valid_area_m2

print(f"Total valid area: {total_valid_area_acres:,.0f} acres")
print(f"Equivalent impervious area: {total_impervious_area_acres:,.0f} acres")
print(f"Watershed-average percent impervious: {percent_impervious_2021:.2f}%")


🤖 **Try asking your AI assistant:** *"I calculated mean percent impervious for a watershed using the NLCD Fractional Impervious Surface raster. Help me interpret this result for H&H modeling, including limitations related to raster resolution, land cover year, and local refinement."*

### Engineering Caution: Check NoData, Do Not Assume It

This module's land cover raster and impervious raster both declare a NoData value in their file metadata, but treating every declared or assumed NoData value the same way (or assuming 0 always means missing) is not safe. Check what a product's official documentation says a background value actually is, and check whether 0 is a legitimate value for that specific product before you mask it out.


## Part 7: Impervious Descriptor - Roads or Other Built Surfaces? 🛣️

The descriptor product is categorical, so it is summarized the same way land cover was in Part 5: count pixels, not average values. Its NoData convention matches the fractional impervious product (127 is the true background; 0 means "not impervious," not missing).


In [ ]:
desc_path = 'nlcd_2021_impervious_descriptor.tif'

with rasterio.open(desc_path) as src:
    desc_clipped, _ = rasterio.mask.mask(
        src, watershed_geom, crop=True, nodata=127
    )

desc_array = desc_clipped[0]
desc_valid = desc_array[desc_array != 127]

desc_values, desc_counts = np.unique(desc_valid, return_counts=True)

descriptor_names = {
    0: 'Not impervious',
    20: 'Primary road',
    21: 'Secondary road',
    22: 'Tertiary road',
    23: 'Thinned road',
    24: 'Non-road built surface',
    25: 'Building (supplemental)',
    26: 'Impervious fill (LCMAP)',
    27: 'Wind turbine',
    28: 'Well pad',
    29: 'Other energy production',
}

total_desc = desc_counts.sum()
for code_val, count in sorted(zip(desc_values, desc_counts), key=lambda x: -x[1]):
    name = descriptor_names.get(int(code_val), f'code {code_val}')
    print(f"  {name:28s} {100*count/total_desc:5.1f}%")


### What This Tells Us

Most of the watershed's area is not impervious at all, matching the land cover result. Of the impervious area that exists, both roads (codes 20 through 23) and other built surfaces (code 24, buildings, driveways, parking) are present. This distinction matters for pollutant loading studies (road surfaces carry different pollutants than rooftops) and for understanding whether stormwater management in this watershed is mostly a transportation right-of-way problem or a private-property problem.


## Part 8: Change Analysis - 2001 versus 2021 📈

### Land Cover Change

Repeat the Part 5 workflow for the 2001 land cover raster, using the same watershed boundary and the same steps.


In [ ]:
lc_2001_path = 'nlcd_2001_land_cover.tif'

with rasterio.open(lc_2001_path) as src:
    lc_2001_clipped, _ = rasterio.mask.mask(src, watershed_geom, crop=True, nodata=0)

lc_2001_array = lc_2001_clipped[0]
class_values_2001, pixel_counts_2001 = np.unique(
    lc_2001_array[lc_2001_array != 0], return_counts=True
)

lc_2001_summary = pd.DataFrame({
    'nlcd_code': class_values_2001,
    'nlcd_class': [nlcd_classes.get(int(c), f'Unknown ({c})') for c in class_values_2001],
    'pixel_count_2001': pixel_counts_2001,
    'percent_watershed_2001': 100 * pixel_counts_2001 / pixel_counts_2001.sum(),
})

# Merge the two years on the class code
#
# An outer merge keeps every class that appears in either year.
# A class that exists in one year but not the other is not "unknown"
# in the missing year: it simply covered 0 percent of the watershed.
# So after merging, fill the missing percentages with 0.
comparison = lc_summary[['nlcd_code', 'percent_watershed']].merge(
    lc_2001_summary[['nlcd_code', 'percent_watershed_2001']], on='nlcd_code', how='outer'
).rename(columns={'percent_watershed': 'percent_watershed_2021'})

comparison['percent_watershed_2001'] = comparison['percent_watershed_2001'].fillna(0)
comparison['percent_watershed_2021'] = comparison['percent_watershed_2021'].fillna(0)

# Look up class names from the legend dictionary, so a class present
# in only one year still gets a proper name
comparison['nlcd_class'] = comparison['nlcd_code'].map(
    lambda c: nlcd_classes.get(int(c), f'Unknown ({c})')
)

comparison['change_pct_points'] = (
    comparison['percent_watershed_2021'] - comparison['percent_watershed_2001']
)
comparison = comparison[['nlcd_code', 'nlcd_class', 'percent_watershed_2001',
                         'percent_watershed_2021', 'change_pct_points']]
comparison = comparison.sort_values('percent_watershed_2021', ascending=False)

print(comparison.round(2).to_string(index=False))


### Percent Impervious Change

Now repeat the Part 6 corrected calculation for the 2001 impervious raster.


In [ ]:
imp_2001_path = 'nlcd_2001_impervious.tif'

with rasterio.open(imp_2001_path) as src:
    imp_2001_clipped, _ = rasterio.mask.mask(src, watershed_geom, crop=True, nodata=127)

imp_2001_array = imp_2001_clipped[0]
imp_2001_valid = imp_2001_array[imp_2001_array != 127].astype(float)

impervious_area_2001_m2 = ((imp_2001_valid / 100) * cell_area_m2).sum()
valid_area_2001_m2 = imp_2001_valid.size * cell_area_m2
percent_impervious_2001 = 100 * impervious_area_2001_m2 / valid_area_2001_m2

print(f"2001 watershed-average percent impervious: {percent_impervious_2001:.2f}%")
print(f"2021 watershed-average percent impervious: {percent_impervious_2021:.2f}%")
print(f"Change over 20 years: {percent_impervious_2021 - percent_impervious_2001:+.2f} percentage points")


### Side-by-Side Maps


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, array, year in zip(axes, [lc_2001_array, lc_array], [2001, 2021]):
    codes_present = sorted(np.unique(array[array != 0]).tolist())
    colors_present = [nlcd_colors[c] for c in codes_present]
    cmap_year = ListedColormap(colors_present)
    idx_map = {c: i for i, c in enumerate(codes_present)}
    display_arr = np.vectorize(lambda v: idx_map.get(v, -1))(array)
    display_arr = np.ma.masked_where(array == 0, display_arr)

    ax.imshow(display_arr, cmap=cmap_year, vmin=0, vmax=len(codes_present) - 1)
    ax.set_title(f"{year} Land Cover", fontweight='bold')
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle(f"{target_watershed['Name'].iloc[0]}: 2001 vs. 2021", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### What This Tells Us

Over 20 years, the developed footprint in this watershed grew, and percent impervious increased along with it. This is precisely the kind of existing-versus-future comparison a stormwater master plan needs: it quantifies how much runoff-generating surface has been added, using an official, repeatable data source rather than a rough visual impression.


## Part 9: Export - Model-Ready Tables and Rasters 📂

Build traceable output tables, following the same documentation habit used in Module 7: source, year, access context, and processing notes on every row.


In [ ]:
import datetime

access_date = datetime.date.today().isoformat()

# Land cover summary, both years, stacked into one table
lc_summary_export = lc_summary.copy()
lc_summary_export['nlcd_year'] = 2021

# Build the 2001 rows from the 2001 counts themselves, so each year's
# area comes from its own raster rather than borrowing the other year's total
lc_2001_export = lc_2001_summary.rename(columns={
    'pixel_count_2001': 'pixel_count',
    'percent_watershed_2001': 'percent_watershed',
}).copy()
lc_2001_export['area_acres'] = lc_2001_export['pixel_count'] * cell_area_acres
lc_2001_export['nlcd_year'] = 2001

lc_export_combined = pd.concat([lc_2001_export, lc_summary_export], ignore_index=True)
lc_export_combined['watershed_id'] = target_huc
lc_export_combined['source'] = 'USGS/MRLC NLCD Land Cover'
lc_export_combined['access_date'] = access_date
lc_export_combined['processing_notes'] = 'Clipped to watershed boundary; NoData=0 excluded'

lc_export_combined.to_csv('watershed_land_cover_summary.csv', index=False)
print(f"Saved watershed_land_cover_summary.csv ({len(lc_export_combined)} rows)")

# Impervious summary, both years
impervious_summary = pd.DataFrame([
    {
        'watershed_id': target_huc,
        'nlcd_year': 2001,
        'mean_impervious_percent': round(percent_impervious_2001, 2),
        'equivalent_impervious_area_acres': round(impervious_area_2001_m2 / 4046.86, 1),
        'total_valid_area_acres': round(valid_area_2001_m2 / 4046.86, 1),
        'source': 'USGS/MRLC NLCD Fractional Impervious Surface',
        'processing_notes': 'NoData=127 (official background value); 0 is a valid reading, not missing data',
        'access_date': access_date,
    },
    {
        'watershed_id': target_huc,
        'nlcd_year': 2021,
        'mean_impervious_percent': round(percent_impervious_2021, 2),
        'equivalent_impervious_area_acres': round(total_impervious_area_acres, 1),
        'total_valid_area_acres': round(total_valid_area_acres, 1),
        'source': 'USGS/MRLC NLCD Fractional Impervious Surface',
        'processing_notes': 'NoData=127 (official background value); 0 is a valid reading, not missing data',
        'access_date': access_date,
    },
])

impervious_summary.to_csv('watershed_impervious_summary.csv', index=False)
print(f"Saved watershed_impervious_summary.csv ({len(impervious_summary)} rows)")

impervious_summary


### Exporting Clipped Rasters

Save the clipped 2021 land cover and impervious rasters as GeoTIFFs, so a GIS user on the project team has the exact analysis extent to work with.


In [ ]:
with rasterio.open(lc_2021_path) as src:
    out_profile = src.profile.copy()
    out_profile.update(
        height=lc_array.shape[0], width=lc_array.shape[1],
        transform=lc_transform, nodata=0, compress='lzw',
    )
    with rasterio.open('clipped_land_cover_2021.tif', 'w', **out_profile) as dst:
        dst.write(lc_array, 1)
        try:
            dst.write_colormap(1, src.colormap(1))
        except ValueError:
            pass

# The impervious export uses its own clip transform (imp_transform, captured
# in Part 6). The two products share one grid here, so the transforms happen
# to match, but each exported raster should always carry the georeferencing
# from its own clip rather than borrowing another raster's.
with rasterio.open(imp_2021_path) as src:
    out_profile = src.profile.copy()
    out_profile.update(
        height=imp_array_v2.shape[0], width=imp_array_v2.shape[1],
        transform=imp_transform, nodata=127, compress='lzw',
    )
    with rasterio.open('clipped_impervious_2021.tif', 'w', **out_profile) as dst:
        dst.write(imp_array_v2, 1)

print("Saved clipped_land_cover_2021.tif and clipped_impervious_2021.tif")


In [ ]:
# Download all output files to your computer
from google.colab import files

files.download('watershed_land_cover_summary.csv')
files.download('watershed_impervious_summary.csv')
files.download('clipped_land_cover_2021.tif')
files.download('clipped_impervious_2021.tif')


### Connecting Forward

This module's land cover table and Module 7's hydrologic soil group table are the two inputs a full composite curve number workflow needs, one pixel-combination at a time (Module 4 built a composite curve number assuming a single soil group; combining it with Module 7's real SSURGO data removes that assumption). The percent impervious value calculated in Part 6 maps directly onto the "percent impervious" parameter used by many HEC-HMS loss methods.


## Part 10: Beyond NLCD - Coastal and Local Alternatives 🔭

### NOAA C-CAP

NOAA's Coastal Change Analysis Program (C-CAP) provides regional and high-resolution land cover products focused on the US coastline. It uses its own classification scheme, its own coverage area, and its own resolution, different from NLCD. For a coastal watershed study, or where a higher-resolution coastal land cover layer is needed, C-CAP is worth evaluating. It is not a replacement for NLCD in general inland CONUS work, which is why this module did not use it.

### Local and Project-Specific Data

NLCD's 30 meter resolution is appropriate for watershed and regional screening, but it is too coarse for parcel-scale design. When a project needs finer detail, consider local municipal land cover or impervious layers, NAIP aerial imagery, or building footprint datasets (the same building footprint data used in Module 3). Whichever source you use, document it the same way this module has documented NLCD: product name, year, source, and access date on every output.

### Source Hierarchy Summary

| If you need... | Use |
|---|---|
| A national or multi-state land cover and impervious baseline | NLCD (this module) |
| Coastal-specific or higher-resolution coastal land cover | NOAA C-CAP |
| Parcel-scale design detail | Local GIS data, NAIP imagery, building footprints |


## Engineering Cautions ⚠️

1. **Categorical rasters hold codes, not quantities.** Never average land cover or descriptor class codes together.
2. **Use the fractional impervious product for percent impervious.** Counting developed land cover classes is not a substitute; it does not account for how impervious each developed pixel actually is.
3. **Verify NoData, do not assume it.** This module found that 0 is NoData for land cover but a legitimate value for the impervious products, where the true background value is 127. Check official documentation for any new product before masking.
4. **Reproject vectors to match the raster CRS, not the reverse.** This keeps the raster's classification and its equal-area property intact.
5. **30 meter resolution is a watershed and regional screening tool, not a parcel-scale design tool.**
6. **Document product, version, and year on every output**, especially for change analyses, since NLCD releases are periodically revised.


## Troubleshooting

| Problem | Likely Cause | What to Try |
|---|---|---|
| Both the live fetch and the fallback fail | No internet connection, or the course repository is temporarily unreachable | Check your connection and rerun the fetch cell; if it persists, try again in a few minutes |
| A fetch is very slow | The watershed's bounding box is unusually large | Reduce `buffer_m`, or confirm the watershed selected is a single HUC-12, not a larger area |
| Clipped raster is empty or tiny | CRS mismatch between vector and raster | Reproject the vector to the raster's CRS before clipping, and print both CRS values to confirm they match |
| Land cover and impervious pixel counts do not match for the same watershed | Wrong NoData value used for one of the products | Check whether 0 is a legitimate value for that product; verify against official documentation before masking |
| Percent impervious looks too high | NoData masking is excluding legitimate low or zero values | Use 127, not 0, as NoData for the fractional impervious and descriptor products |
| Areas do not match a GIS calculation | A geographic (degree-based) CRS was used for area math | Confirm you are working in the raster's native equal-area CRS (EPSG:5070 for NLCD) |
| Map colors look wrong or classes are missing from the legend | The legend dictionary does not include every code actually present | Print `np.unique()` on the array and confirm every value has an entry in the legend dictionary |
| Colab upload fails | The watershed ZIP was not selected | Re-run the upload cell and select the watershed file |


## Practice Exercises 🎯

Each exercise can be completed by lightly editing code that already appears in this notebook.

### Exercise 1: A Different Watershed

Change `target_huc` in Part 4 to a different HUC-12 from the same file (try `'101900090106'` or `'101900090109'`). Because this notebook fetches its own rasters, you will need to rerun the bounding box, fetch, and reprojection steps for the new watershed before rerunning the Part 5 summary; the automated fetch means this works for any watershed in the file, not just the one used throughout this lesson.


In [ ]:
# EXERCISE 1: A different watershed
# Your code here.
#
# Steps:
# 1. Set target_huc to a different HUC-12 code from the watersheds file.
# 2. Re-select target_watershed.
# 3. Recompute the bounding box (minx, miny, maxx, maxy) for the new watershed.
# 4. Call fetch_nlcd_raster() again for each of the 5 products, same as Part 4.
# 5. Reproject the new target_watershed to the raster's CRS.
# 6. Rerun the Part 5 clip-and-summarize steps.
# 7. Print the new land cover summary table.


### Exercise 2: Impervious Hotspots

Using the corrected `imp_array_v2` array from Part 6, count how many cells have a percent impervious value above 50, and convert that count to acres.


In [ ]:
# EXERCISE 2: Impervious hotspots
# Your code here.
#
# Steps:
# 1. Build a boolean mask: imp_array_v2 > 50, excluding the 127 background value.
# 2. Count how many cells meet that condition.
# 3. Multiply by cell_area_acres to get the hotspot area.


### Exercise 3: Descriptor Road Percentage

Using the Part 7 descriptor results, calculate what percentage of the *impervious* area (not the whole watershed) is made up of roads (codes 20 through 23) versus non-road built surfaces (code 24 and higher).


In [ ]:
# EXERCISE 3: Descriptor road percentage
# Your code here.
#
# Steps:
# 1. From desc_values and desc_counts, sum the counts for codes 20-23 (roads).
# 2. Sum the counts for codes 24 and above (non-road built surfaces).
# 3. Divide each by the total impervious count (excluding code 0) and print as percentages.


### Challenge Exercise: A Reusable Summary Function (AI-Assisted)

Ask your AI assistant to help you wrap the Part 5 land cover workflow into a function, `summarize_land_cover(huc12_code, raster_path)`, that returns the summary DataFrame for any watershed and any year's raster. Test it on two different watersheds.

Suggested prompt to paste into your AI assistant:

*"Help me write a Python function called summarize_land_cover(huc12_code, raster_path) that selects a watershed by HUC-12 code from my watersheds GeoDataFrame, reprojects it to match the raster's CRS, clips the raster, and returns a summary DataFrame with nlcd_code, nlcd_class, pixel_count, area_acres, and percent_watershed columns, reusing the nlcd_classes dictionary already defined in my notebook."*


In [ ]:
# CHALLENGE EXERCISE: Build summarize_land_cover(huc12_code, raster_path)
# Your code here.
#
# Steps:
# 1. Ask your AI assistant using the suggested prompt above.
# 2. Review the function. Does it reuse the nlcd_classes dictionary?
#    Does it handle a watershed with no matching HUC-12 code?
# 3. Test the function on two different watersheds and print both results.


## That's Module 8 Done!

Think about what this module walked through. A land cover summary that used to mean opening a GIS desktop application, clipping a national raster by hand, and tallying pixels in a separate spreadsheet now runs in a script. More importantly, this module caught a real, easy-to-miss mistake along the way: trusting a raster's declared NoData value without checking whether it actually applies to that product. That habit, verify before you trust, is worth more than any single line of code in this notebook.

### What you can do now

- **Tell categorical and continuous rasters apart**, and choose the right summary method for each.
- **Clip and summarize land cover** by area and percent, with a built-in sanity check against the watershed's true area.
- **Calculate percent impervious correctly**, including catching a NoData mistake that would otherwise inflate the result by a wide margin.
- **Separate roads from other impervious surfaces** using the descriptor product.
- **Quantify land cover change** between two years using the same workflow twice.

### Next Steps

- **Combine with Modules 4 and 7.** This module's land cover table and Module 7's hydrologic soil group table are the two real inputs to a proper composite curve number calculation, replacing Module 4's single-soil-group assumption.
- **On your own.** Run this workflow on a watershed from a current project, and compare the percent impervious result against any as-built or GIS-based estimate you already have.
- **Coastal or parcel-scale work.** Revisit Part 10 for when NLCD is not the right resolution.

The full repository, including data files and the rest of the course, is at [github.com/mohsennasab/python-fundamentals-hh](https://github.com/mohsennasab/python-fundamentals-hh).
